I used this model to test out building and training the model. I used colab's GPUs.

In [ ]:
# %pip install transformers h5py pandas pyarrow protobuf

In [ ]:
# %pip install torch

# install cuda on windows
# %pip install torch --index-url https://download.pytorch.org/whl/cu126

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
from pathlib import Path
from typing import List, Optional

import h5py
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset


# sources where the first few turns can be text-only (hdf5_idx == -1)
_TEXT_ONLY_SOURCES = {"styletalk"}

# styletalk always has exactly this many text-only turns at the front of a chain
_STYLETALK_TEXT_ONLY_TURNS = 3


class AudioHDF5Dataset(Dataset):
    """
    Streams waveforms from HDF5 alongside metadata. Each item is a
    conversation chain of `num_turns` utterances in chronological order,
    resolved by walking the prev_filename linked list backwards from the
    anchor utterance (turn_index == num_turns-1 ) (zero indexed).

    Rows missing audio (hdf5_idx == -1) are excluded entirely, along with
    any chain that can't be fully resolved due to broken links.
    """

    def __init__(
        self,
        h5_path:      str,
        meta_path:    str,
        meta_columns: Optional[List[str]] = None,
        max_len_sec:  Optional[float]     = None,
        sample_rate:  int                 = 16_000,
        num_turns:    int                 = 5,
        transform=None,
    ):
        self.h5_path    = Path(h5_path)
        self.meta_path  = Path(meta_path)
        self.transform  = transform
        self.sr         = sample_rate
        self.max_len    = int(max_len_sec * sample_rate) if max_len_sec else None
        self.num_turns  = num_turns

        meta = pd.read_parquet(meta_path)

        # drop hard errors only; text-only rows (-1) are handled per-source below
        meta = meta[meta["hdf5_idx"] >= -1].reset_index(drop=True)

        # index by relative_audio_path so prev_filename lookups are O(1)
        self._path_to_row = meta.set_index("relative_audio_path")

        # anchor rows are the "last" turn in each chain we want to load
        anchors = meta[meta["turn_index"] == (num_turns - 1)].reset_index(drop=True)

        # resolve every anchor into a full chain, drop any with broken links
        if meta_columns is not None:
            # always keep the fields we need for chain walking + audio loading
            required = {"hdf5_idx", "relative_audio_path", "prev_filename", "turn_index"}
            self._extra_cols = [c for c in meta_columns if c not in required]
        else:
            required = {"hdf5_idx", "relative_audio_path", "prev_filename", "turn_index"}
            self._extra_cols = [c for c in meta.columns if c not in required]

        self._chains = self._build_chains(anchors)
        self._h5 = None  # lazy-open per worker so forking doesn't break the fd

    def _build_chains(self, anchors: pd.DataFrame) -> List[List[dict]]:
        chains = []
        for _, anchor_row in anchors.iterrows():
            chain = self._walk_chain(anchor_row)
            if chain is not None:
                chains.append(chain)
        return chains

    def _walk_chain(self, anchor_row) -> Optional[List[dict]]:
        # walk backwards num_turns times, collecting rows oldest-first
        chain = []
        current = anchor_row

        for _ in range(self.num_turns):
            chain.append(current)
            prev_path = current.get("prev_filename")

            # stop early if we're at the start of a conversation
            if pd.isna(prev_path) or prev_path == "":
                break

            if prev_path not in self._path_to_row.index:
                # broken link -- skip this chain entirely
                return None

            current = self._path_to_row.loc[prev_path]

        # reverse so the chain runs chronologically oldest -> newest
        chain.reverse()

        # only keep full chains -- shorter ones are incomplete conversations
        if len(chain) < self.num_turns:
            return None

        source = str(anchor_row.get("source", "")).lower()

        if source == "styletalk":
            return self._validate_styletalk_chain(chain)
        else:
            return self._validate_expresso_chain(chain)


    def _validate_styletalk_chain(self, chain: list) -> Optional[list]:
        # styletalk: first N turns must be text-only, the rest must have audio
        for turn_pos, row in enumerate(chain):
            hdf5_idx = int(row["hdf5_idx"])
            is_text_only_slot = turn_pos < _STYLETALK_TEXT_ONLY_TURNS

            if is_text_only_slot and hdf5_idx != -1:
                # we expect text-only here; a real audio file is unexpected but not fatal
                pass
            elif not is_text_only_slot and hdf5_idx == -1:
                # audio turns must actually have audio
                return None

        return chain

    def _validate_expresso_chain(self, chain: list) -> Optional[list]:
        # expresso: no text-only rows allowed anywhere in the chain
        for row in chain:
            if int(row["hdf5_idx"]) == -1:
                return None
        return chain

    def _get_h5(self):
        # open lazily so each DataLoader worker gets its own file handle
        if self._h5 is None:
            self._h5 = h5py.File(self.h5_path, "r", swmr=True)
        return self._h5

    def _load_waveform(self, hdf5_idx: int):
        hf  = self._get_h5()
        key = f"audio/{hdf5_idx:06d}"
        ds  = hf[key]
        wav = ds[()]
        sr  = int(ds.attrs["sample_rate"])
        return wav, sr

    def _pad_or_trim(self, wav: np.ndarray) -> np.ndarray:
        if self.max_len is None:
            return wav
        if len(wav) >= self.max_len:
            return wav[: self.max_len]
        return np.pad(wav, (0, self.max_len - len(wav)))

    def __len__(self):
        return len(self._chains)

    def __getitem__(self, i):
        chain = self._chains[i]

        utterances = []
        for row in chain:
            hdf5_idx  = int(row["hdf5_idx"])
            is_text_only = hdf5_idx == -1

            if is_text_only:
                # no waveform to load; downstream should check this flag
                wav_tensor  = None
                sample_rate = self.sr
            else:
                wav, sample_rate = self._load_waveform(hdf5_idx)

                if self.transform is not None:
                    wav = self.transform(wav)

                wav        = self._pad_or_trim(wav)
                wav_tensor = torch.from_numpy(wav)

            utt = {
                "audio":               wav_tensor,
                "sample_rate":         sample_rate,
                "hdf5_idx":            hdf5_idx,
                "turn_index":          int(row["turn_index"]),
                "text_only":           is_text_only,
                "relative_audio_path": row.name,  # relative_audio_path is the index
                "speaker_id" : row["speakerid"]
            }

            for col in self._extra_cols:
                val = row.get(col, "")
                if isinstance(val, float) and np.isnan(val):
                    val = ""
                utt[col] = val

            utterances.append(utt)

        return utterances  # list of dicts, chronological order

    def __del__(self):
        if self._h5 is not None:
            try:
                self._h5.close()
            except Exception:
                pass


def collate_pad(batch):
    """
    Collates a batch of conversation chains. Each item in `batch` is a list
    of utterance dicts (one per turn). Returns a dict where 'audio' is
    (B, T, max_wav_len), 'lengths' is (B, T), and 'text_only' is (B, T) bool.
    Text-only turns (audio=None) are left as zero-padded rows in the tensor.

    Use as: DataLoader(..., collate_fn=collate_pad)
    """
    B = len(batch)
    T = len(batch[0])

    # only consider turns that actually have audio when computing max length
    audio_lengths = [
        utt["audio"].shape[0]
        for chain in batch
        for utt in chain
        if utt["audio"] is not None
    ]
    max_len = max(audio_lengths) if audio_lengths else 0

    padded    = torch.zeros(B, T, max_len, dtype=torch.float32)
    lengths   = torch.zeros(B, T, dtype=torch.long)
    text_only = torch.zeros(B, T, dtype=torch.bool)

    for b, chain in enumerate(batch):
        for t, utt in enumerate(chain):
            if utt["audio"] is not None:
                L = utt["audio"].shape[0]
                padded[b, t, :L] = utt["audio"]
                lengths[b, t]    = L
            text_only[b, t] = utt["text_only"]

    meta_keys = [k for k in batch[0][0] if k not in ("audio", "text_only")]
    out = {k: [[utt[k] for utt in chain] for chain in batch] for k in meta_keys}

    out["audio"]     = padded     # (B, T, max_wav_len)
    out["lengths"]   = lengths    # (B, T)
    out["text_only"] = text_only  # (B, T)
    return out

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


from torch.utils.data import DataLoader
# from ConvStyleDataset import AudioHDF5Dataset, collate_pad

H5_PATH     = "/content/drive/MyDrive/capstone/data/merged_audio_500.h5" # <--- UPDATE THIS PATH
META_PATH   = "/content/drive/MyDrive/capstone/data/merged_metadata_500.parquet" # <--- UPDATE THIS PATH
# H5_PATH     = "../data_TEMP/merged_audio_500.h5" # <--- UPDATE THIS PATH
# META_PATH   = "../data_TEMP/merged_metadata_500.parquet" # <--- UPDATE THIS PATH
BATCH_SIZE = 1      # Reduced batch size for memory optimization. Try 1 first. <--- MODIFIED
SAMPLE_RATE = 16_000
MAX_NUM_TURNS = 5

dataset = AudioHDF5Dataset(
    h5_path=H5_PATH,
    meta_path=META_PATH,
    meta_columns=["transcription"],
    sample_rate=SAMPLE_RATE,
    num_turns=MAX_NUM_TURNS # Changed from 5 to 1 to allow loading single-turn utterances for diagnosis
    # max_len_sec=10.0, # Uncomment and adjust this value (e.g., 10 seconds) for further memory optimization.
    # no max_len_sec -- collate_pad handles variable lengths
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_pad,
    num_workers=0,
)

Mounted at /content/drive


In [ ]:
# data batch test
batch = next(iter(loader))
print(batch.keys())
audio   = batch["audio"]           # (B, T, max_wav_len)
lengths = batch["lengths"]         # (B, T)
texts   = batch["transcription"]   # list[B][T]
wav_files = batch["relative_audio_path"]
speakers = batch["speaker_id"]

B, T, max_wav_len = audio.shape

print(f"audio shape   : {tuple(audio.shape)}  (batch, turns, samples)")
print(f"audio dtype   : {audio.dtype}")
print(f"lengths shape : {tuple(lengths.shape)}")
print()

# print each chain as a conversation so ordering is easy to eyeball
for b in range(B):
    print(f"Chain {b}:")
    for t in range(T):
        L     = lengths[b, t].item()
        dur_s = L / dataset.sr
        text  = texts[b][t]
        file = wav_files[b][t]
        spker = speakers[b][t]
        display = text[:77] + "..." if len(text) > 80 else text
        print(f"Spk {spker}, turn {t+1}  |  {L} samples ({dur_s:.2f}s)  |  {display!r}  | {file}")
    print()

# verify no padding bleeds into real audio for any turn
for b in range(B):
    for t in range(T):
        L    = lengths[b, t].item()
        tail = audio[b, t, L:]
        if tail.numel() > 0:
            assert tail.abs().max() == 0, f"chain {b} turn {t}: non-zero past length {L}"

print("Padding check passed")

dict_keys(['sample_rate', 'hdf5_idx', 'turn_index', 'relative_audio_path', 'speaker_id', 'transcription', 'audio', 'lengths', 'text_only'])
audio shape   : (1, 5, 580960)  (batch, turns, samples)
audio dtype   : torch.float32
lengths shape : (1, 5)

Chain 0:
Spk ex02, turn 1  |  91040 samples (5.69s)  |  ' These are all examples of different states of human emotion.'  | audio_48khz/conversational_vad_segmented/ex04-ex02/enunciated/ex04-ex02_enunciated_001_channel2_segment_585.99_591.68.wav
Spk ex04, turn 2  |  127200 samples (7.95s)  |  ' And when you experience euphoria, what does that look like in your brain?'  | audio_48khz/conversational_vad_segmented/ex04-ex02/enunciated/ex04-ex02_enunciated_001_channel1_segment_593.35_601.3.wav
Spk ex02, turn 3  |  580960 samples (36.31s)  |  ' it would depend on the circumstance. Sometimes euphoria can come from everyt...'  | audio_48khz/conversational_vad_segmented/ex04-ex02/enunciated/ex04-ex02_enunciated_001_channel2_segment_601.43_637.74.wav

In [ ]:
# grab one chain directly from the dataset (no collation needed for inspection)
chain_idx = 0
chain = dataset[chain_idx]

print(f"Total chains in dataset: {len(dataset)}  (num_turns={dataset.num_turns})")
print(f"\nChain {chain_idx}  ({len(chain)} turns)")
print("-" * 60)

for utt in chain:
    dur_s = utt["audio"].shape[0] / utt["sample_rate"]
    print(f"spker {utt["speaker_id"]} |   turn {utt['turn_index']}  |  hdf5_idx={utt['hdf5_idx']}  |  {utt['audio'].shape[0]} samples ({dur_s:.2f}s)")
    for k, v in utt.items():
        if k in {"audio", "sample_rate", "hdf5_idx", "turn_index"}:
            continue
        display = str(v)
        if len(display) > 80:
            display = display[:77] + "..."
        print(f"    {k}: {display}")

print("-" * 60)

Total chains in dataset: 100  (num_turns=5)

Chain 0  (5 turns)
------------------------------------------------------------
spker ex01 |   turn 0  |  hdf5_idx=0  |  63360 samples (3.96s)
    text_only: False
    relative_audio_path: audio_48khz/conversational_vad_segmented/ex01-ex02/fast/ex01-ex02_fast_003_ch...
    speaker_id: ex01
    transcription:  I like sticking little toothpicks in them and making them like soldiers.
spker ex02 |   turn 1  |  hdf5_idx=1  |  34240 samples (2.14s)
    text_only: False
    relative_audio_path: audio_48khz/conversational_vad_segmented/ex01-ex02/fast/ex01-ex02_fast_003_ch...
    speaker_id: ex02
    transcription:  Yeah, and we walk them around on the table.
spker ex01 |   turn 2  |  hdf5_idx=2  |  137920 samples (8.62s)
    text_only: False
    relative_audio_path: audio_48khz/conversational_vad_segmented/ex01-ex02/fast/ex01-ex02_fast_003_ch...
    speaker_id: ex01
    transcription:  Yeah, it's very cute. You roll them around. They, you know, I do

In [ ]:
from transformers import BertTokenizer, BertModel, WavLMModel, Wav2Vec2FeatureExtractor

bert_tokenizer  = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model      = BertModel.from_pretrained("bert-base-uncased").half().to(device)  # using half() to limit gpu memory for quick tests

# WavLM only needs a feature extractor, not a full processor with tokenizer
wavlm_processor = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
wavlm_model     = WavLMModel.from_pretrained("microsoft/wavlm-base-plus").half().to(device)

for model in (bert_model, wavlm_model):
    for param in model.parameters():
        param.requires_grad = False
    model.eval()

print("Encoders loaded and frozen")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

Encoders loaded and frozen


In [ ]:
class SelfAttentivePooling(nn.Module):

    def __init__(self, dim: int):
        super().__init__()
        self.scorer = nn.Linear(dim, 1, bias=False)

    def forward(self, hidden: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        # hidden: (B, T, d),  mask: (B, T) -- 1 for real tokens, 0 for padding
        scores = self.scorer(hidden).squeeze(-1)  # (B, T)

        if mask is not None:
            # push padding positions to -inf so softmax zeroes them out
            scores = scores.masked_fill(~mask.bool(), float("-inf"))

        weights = F.softmax(scores, dim=-1)
        return (weights.unsqueeze(-1) * hidden).sum(dim=1)  # (B, d)

In [ ]:
class ModalityEncoder(nn.Module):

    def __init__(self, backbone, pooler):
        super().__init__()
        self.backbone = backbone
        self.pooler   = pooler

    def forward(self, hidden: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        return self.pooler(hidden, mask)  # (B, d)

In [ ]:
def encode_text_inputs(texts, tokenizer):
    return tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    ).to(device)

def encode_audio_inputs(audio, lengths, processor):
    inputs = processor(
        list(audio.cpu().numpy()),
        sampling_rate=SAMPLE_RATE,
        return_tensors="pt",
        padding=True,
    ).to(device)
    return inputs, lengths

In [ ]:
class DualModalityEmbedder(nn.Module):

    def __init__(self, text_encoder, audio_encoder, tokenizer, processor):
        super().__init__()
        self.text_encoder  = text_encoder
        self.audio_encoder = audio_encoder
        self.tokenizer     = tokenizer
        self.processor     = processor

    def embed_text(self, texts):
        # texts is list[B][T] -- flatten to a single batch so the encoder sees B*T items
        B = len(texts)
        T = len(texts[0])
        flat_texts = [texts[b][t] for b in range(B) for t in range(T)]

        tokens = encode_text_inputs(flat_texts, self.tokenizer)
        with torch.no_grad():
            hidden = self.text_encoder.backbone(**tokens).last_hidden_state
        flat_emb = self.text_encoder(hidden, tokens["attention_mask"])  # (B*T, d)

        return flat_emb.view(B, T, -1)  # (B, T, d)

    def embed_audio(self, audio, lengths, text_only=None):
        # audio: (B, T, samples), lengths: (B, T)
        B, T, _ = audio.shape
        d_model  = 768

        out = torch.zeros(B, T, d_model, dtype=audio.dtype, device=audio.device)

        for t in range(T):
            # skip turns where every item in the batch has no audio
            if text_only is not None and text_only[:, t].all():
                continue

            audio_t   = audio[:, t, :]    # (B, samples)
            lengths_t = lengths[:, t]     # (B,)

            inputs, lengths_t = encode_audio_inputs(audio_t, lengths_t, self.processor)
            with torch.no_grad():
                hidden = self.audio_encoder.backbone(**inputs).last_hidden_state

            T_out    = hidden.shape[1]
            raw_mask = (torch.arange(audio_t.shape[1], device=device).unsqueeze(0)
                        < lengths_t.unsqueeze(1).to(device)).float()
            mask_ds  = F.interpolate(raw_mask.unsqueeze(1), size=T_out, mode="nearest").squeeze(1).bool()

            emb = self.audio_encoder(hidden, mask_ds)  # (B, d)
            out[:, t, :] = emb

        return out  # (B, T, d)

    def forward(self, audio, lengths, texts, text_only=None):
        text_emb  = self.embed_text(texts)
        audio_emb = self.embed_audio(audio, lengths, text_only)
        return text_emb, audio_emb  # both (B, T, d)


# CHANGED TO HALF
text_encoder  = ModalityEncoder(bert_model,  SelfAttentivePooling(768).half()).to(device)
audio_encoder = ModalityEncoder(wavlm_model, SelfAttentivePooling(768).half()).to(device)

embedder = DualModalityEmbedder(
    text_encoder, audio_encoder, bert_tokenizer, wavlm_processor
).to(device)

print("Embedder ready")

Embedder ready


In [ ]:
embedder.eval()

# Re-define batch here as it was not found in the kernel state
batch = next(iter(loader))

text_emb, audio_emb = embedder(
    batch["audio"].to(device).half(),
    batch["lengths"],
    batch["transcription"],
    text_only=batch["text_only"].to(device),
)

print("text_emb shape :", text_emb.shape)   # (B, T, 768)
print("audio_emb shape:", audio_emb.shape)  # (B, T, 768)


assert text_emb.shape  == (BATCH_SIZE, MAX_NUM_TURNS, 768)
assert audio_emb.shape == (BATCH_SIZE, MAX_NUM_TURNS, 768)
assert not torch.allclose(text_emb, audio_emb), "embeddings are suspiciously identical"

print("All assertions passed")

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


text_emb shape : torch.Size([1, 5, 768])
audio_emb shape: torch.Size([1, 5, 768])
All assertions passed


In [ ]:
class ContextAwareTransformer(nn.Module):
    def __init__(self, d_model: int, nhead: int, num_encoder_layers: int, dim_feedforward: int, dropout: float = 0.1):
        super().__init__()
        # TransformerEncoderLayer is the basic building block for the encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True # Set batch_first to True for (B, S, E) input
        )
        # TransformerEncoder stacks multiple EncoderLayers
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_encoder_layers)

    def forward(self, turn_embeddings: torch.Tensor):
        # Input `turn_embeddings` is expected to be (Batch_Size, Num_Turns, Embedding_Dim) which is (B, S, E)

        # Generate causal mask: ensures each position only attends to previous positions
        seq_len = turn_embeddings.shape[1] # Sequence length is the second dimension for batch_first=True
        src_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(turn_embeddings.device)

        # Pass through the Transformer Encoder with the mask
        output = self.transformer_encoder(turn_embeddings, mask=src_mask)

        # Output is already (Batch_Size, Num_Turns, Embedding_Dim) because batch_first=True
        return output

In [ ]:
# Define transformer parameters
d_model = 768 # Embedding dimension from BERT/WavLM
nhead = 8 # Number of attention heads
num_encoder_layers = 2 # Number of transformer encoder layers
dim_feedforward = 2048 # Dimension of the feedforward network model

# Instantiate the ContextAwareTransformer and move it to the device
context_transformer = ContextAwareTransformer(
    d_model=d_model,
    nhead=nhead,
    num_encoder_layers=num_encoder_layers,
    dim_feedforward=dim_feedforward
).to(device).half()

# pass audio and text embeddings separately
context_aware_embeddings_audio = context_transformer(audio_emb)
context_aware_embeddings_text = context_transformer(text_emb)

print("Context-aware audio embeddings shape:", context_aware_embeddings_audio.shape)
print("Context-aware audio embeddings shape:", context_aware_embeddings_text.shape)
assert context_aware_embeddings_audio.shape == (BATCH_SIZE, MAX_NUM_TURNS, d_model)
assert context_aware_embeddings_text.shape == (BATCH_SIZE, MAX_NUM_TURNS, d_model)
print("Context-aware audio transformer output shape assertion passed.")

Context-aware audio embeddings shape: torch.Size([1, 5, 768])
Context-aware audio embeddings shape: torch.Size([1, 5, 768])
Context-aware audio transformer output shape assertion passed.


In [ ]:
import torch.nn.functional as F
from typing import List, Optional

class IntraSpeakerTransformer(nn.Module):
    def __init__(self, d_model: int, nhead: int, num_encoder_layers: int
                 , dim_feedforward: int, dropout: float = 0.1):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_encoder_layers)

    def forward(self, turn_embeddings: torch.Tensor, speaker_ids_list: List[List[str]]):
        B, seq_len, _ = turn_embeddings.shape
        device = turn_embeddings.device

        # Generate causal mask and ensure it has the same dtype as turn_embeddings
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(device).to(turn_embeddings.dtype)

        # Create numerical speaker IDs for the current batch
        numerical_speaker_ids = []
        global_speaker_id_map = {}
        next_id = 0

        for b_idx in range(B):
            chain_speakers = speaker_ids_list[b_idx]
            chain_numerical_ids = []
            for speaker_str in chain_speakers:
                if speaker_str not in global_speaker_id_map:
                    global_speaker_id_map[speaker_str] = next_id
                    next_id += 1
                chain_numerical_ids.append(global_speaker_id_map[speaker_str])
            numerical_speaker_ids.append(chain_numerical_ids)

        # Convert to tensor (B, seq_len)
        speaker_ids_tensor = torch.tensor(numerical_speaker_ids, dtype=torch.long, device=device)

        # Create same speaker boolean mask (B, seq_len, seq_len)
        same_speaker_bool_mask = (speaker_ids_tensor.unsqueeze(2) == speaker_ids_tensor.unsqueeze(1))

        # Intra-speaker mask: allow attention within the same speaker, block attention to different speakers
        # Additive mask: 0 for allowed, -inf for blocked
        inf_val = torch.finfo(turn_embeddings.dtype).min
        # Explicitly cast 0. to the correct dtype for torch.where
        zero_tensor = torch.tensor(0., dtype=turn_embeddings.dtype, device=device)
        intra_speaker_additive_mask = torch.where(same_speaker_bool_mask, zero_tensor, inf_val)

        # Combine causal mask with intra-speaker mask
        # Causal mask is (S, S), intra_speaker_additive_mask is (B, S, S)
        # Unsqueezing causal_mask to (1, S, S) allows broadcasting
        final_mask = causal_mask.unsqueeze(0) + intra_speaker_additive_mask

        # Reshape the mask to (B * nhead, S, S)
        nhead_local = self.transformer_encoder.layers[0].self_attn.num_heads
        final_mask = final_mask.unsqueeze(1).expand(-1, nhead_local, -1, -1).reshape(B * nhead_local, seq_len, seq_len)

        output = self.transformer_encoder(turn_embeddings, mask=final_mask)
        return output

In [ ]:
import torch.nn.functional as F
from typing import List, Optional

class InterSpeakerTransformer(nn.Module):
    def __init__(self, d_model: int, nhead: int, num_encoder_layers: int
                 , dim_feedforward: int, dropout: float = 0.1):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_encoder_layers)

    def forward(self, turn_embeddings: torch.Tensor, speaker_ids_list: List[List[str]]):
        B, seq_len, _ = turn_embeddings.shape
        device = turn_embeddings.device

        # Generate causal mask and ensure it has the same dtype as turn_embeddings
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(device).to(turn_embeddings.dtype)

        # Create numerical speaker IDs for the current batch
        numerical_speaker_ids = []
        global_speaker_id_map = {}
        next_id = 0

        for b_idx in range(B):
            chain_speakers = speaker_ids_list[b_idx]
            chain_numerical_ids = []
            for speaker_str in chain_speakers:
                if speaker_str not in global_speaker_id_map:
                    global_speaker_id_map[speaker_str] = next_id
                    next_id += 1
                chain_numerical_ids.append(global_speaker_id_map[speaker_str])
            numerical_speaker_ids.append(chain_numerical_ids)

        # Convert to tensor (B, seq_len)
        speaker_ids_tensor = torch.tensor(numerical_speaker_ids, dtype=torch.long, device=device)

        # Create same speaker boolean mask (B, seq_len, seq_len)
        same_speaker_bool_mask = (speaker_ids_tensor.unsqueeze(2) == speaker_ids_tensor.unsqueeze(1))

        # Inter-speaker mask: allow attention between different speakers, block attention to the same speaker
        # Additive mask: 0 for allowed, -inf for blocked
        inf_val = torch.finfo(turn_embeddings.dtype).min
        # Explicitly cast 0. to the correct dtype for torch.where
        zero_tensor = torch.tensor(0., dtype=turn_embeddings.dtype, device=device)
        inter_speaker_additive_mask = torch.where(~same_speaker_bool_mask, zero_tensor, inf_val)

        # Combine causal mask with inter-speaker mask
        final_mask = causal_mask.unsqueeze(0) + inter_speaker_additive_mask

        # Reshape the mask to (B * nhead, S, S)
        nhead_local = self.transformer_encoder.layers[0].self_attn.num_heads
        final_mask = final_mask.unsqueeze(1).expand(-1, nhead_local, -1, -1).reshape(B * nhead_local, seq_len, seq_len)

        output = self.transformer_encoder(turn_embeddings, mask=final_mask)
        return output

In [ ]:
class SpeakerAwareTransformer(nn.Module):
    def __init__(self, d_model: int, nhead: int, num_encoder_layers: int, dim_feedforward: int, dropout: float = 0.1):
        super().__init__()
        # Instantiate separate ContextAwareTransformer for intra-speaker masking
        self.intra_transformer = IntraSpeakerTransformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout
        )
        # Instantiate separate ContextAwareTransformer for inter-speaker masking
        self.inter_transformer = InterSpeakerTransformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout
        )

    def forward(self, turn_embeddings: torch.Tensor, speaker_ids_list: List[List[str]]):
        # Pass through the intra-speaker context aware transformer
        intra_emb = self.intra_transformer(
            turn_embeddings,
            speaker_ids_list=speaker_ids_list
        )

        # Pass through the inter-speaker context aware transformer
        inter_emb = self.inter_transformer(
            turn_embeddings,
            speaker_ids_list=speaker_ids_list
        )
        return intra_emb, inter_emb

In [ ]:
# Instantiate the SpeakerAwareTransformer and move it to the device
speaker_aware_transformer = SpeakerAwareTransformer(
    d_model=d_model,
    nhead=nhead,
    num_encoder_layers=num_encoder_layers,
    dim_feedforward=dim_feedforward
).to(device).half()

# Pass audio embeddings through the speaker-aware transformer
intra_speaker_aware_audio_emb, inter_speaker_aware_audio_emb = speaker_aware_transformer(
    audio_emb,
    speaker_ids_list=batch["speaker_id"]
)

# Pass text embeddings through the speaker-aware transformer
intra_speaker_aware_text_emb, inter_speaker_aware_text_emb = speaker_aware_transformer(
    text_emb,
    speaker_ids_list=batch["speaker_id"]
)

print("Intra-speaker aware audio embeddings shape:", intra_speaker_aware_audio_emb.shape)
print("Inter-speaker aware audio embeddings shape:", inter_speaker_aware_audio_emb.shape)
print("Intra-speaker aware text embeddings shape:", intra_speaker_aware_text_emb.shape)
print("Inter-speaker aware text embeddings shape:", inter_speaker_aware_text_emb.shape)

assert intra_speaker_aware_audio_emb.shape == (BATCH_SIZE, MAX_NUM_TURNS, d_model)
assert inter_speaker_aware_audio_emb.shape == (BATCH_SIZE, MAX_NUM_TURNS, d_model)
assert intra_speaker_aware_text_emb.shape == (BATCH_SIZE, MAX_NUM_TURNS, d_model)
assert inter_speaker_aware_text_emb.shape == (BATCH_SIZE, MAX_NUM_TURNS, d_model)
print("Speaker-aware transformer output shapes assertion passed.")

Intra-speaker aware audio embeddings shape: torch.Size([1, 5, 768])
Inter-speaker aware audio embeddings shape: torch.Size([1, 5, 768])
Intra-speaker aware text embeddings shape: torch.Size([1, 5, 768])
Inter-speaker aware text embeddings shape: torch.Size([1, 5, 768])
Speaker-aware transformer output shapes assertion passed.
